In [2]:
import pandas as pd

In [3]:
df_market = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/merged_market_features.csv")

df_income = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/financial_features/income_statement_features.csv")

df_balancesheet = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/financial_features/balance_feature.csv")

df_cashflow = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/financial_features/cashflow_features.csv")

In [5]:
print("---- MARKET ----")
print(df_market.columns.to_list())

print("\n---- INCOME ----")
print(df_income.columns.to_list())

print("\n---- BALANCE SHEET ----")
print(df_balancesheet.columns.to_list())

print("\n---- CASH FLOW ----")
print(df_cashflow.columns.to_list())


---- MARKET ----
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 't1', 'Price at t1', 'Future Return', 'Target', 'Historical Return', 'Historical Volatility', 'Average Trading Volume']

---- INCOME ----
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'Revenue Growth', 'Gross Profit Margin', 'Operating Profit Margin', 'Net Profit Margin', 'Operating Income Growth', 'Net Income Growth']

---- BALANCE SHEET ----
['Ticker', 'SimFinId', 'Fiscal Year', 'Cash, Cash Equivalents & Short Term Investments', 'Total Current Assets', 'Total Assets', 'Total Current Liabilities', 'Total Liabilities', 'Total Equity', 'balance_diff', 'balance_error_pct', 'Short Term Debt', 'Long Term Debt', 'Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']

---- CASH FLOW ----
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 'Operating Cash Flow', 'Capital Expenditure', 'Re

## Merge Market and Income

In [7]:
income_merge = df_income[['SimFinId', 'Fiscal Year', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'Revenue Growth', 'Gross Profit Margin', 'Operating Profit Margin', 'Net Profit Margin', 'Operating Income Growth', 'Net Income Growth']]

df_merged = df_market.merge(income_merge, on=["SimFinId", "Fiscal Year"], how="left", validate="one_to_one")

In [8]:
print(df_merged.shape)

print("Duplicate company-year rows:", df_merged.duplicated(["SimFinId", "Fiscal Year"]).sum())

(386, 24)
Duplicate company-year rows: 0


## Add Cash Flow

In [10]:
cashflow_merge = df_cashflow[['SimFinId', 'Fiscal Year', 'Operating Cash Flow', 'Capital Expenditure', 'Revenue', 'Free Cash Flow', 'Operating Cash Flow Growth', 'Free Cash Flow Growth', 'FCF Margin']]

df_merged = df_merged.merge(cashflow_merge, on=["SimFinId", "Fiscal Year"], how="left", validate="one_to_one")

In [15]:
df_merged = df_merged.drop(columns=["Revenue_y"])
df_merged = df_merged.rename(columns={"Revenue_x": "Revenue"})

print(df_merged.shape)
print(df_merged.columns.to_list())

(386, 30)
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 't1', 'Price at t1', 'Future Return', 'Target', 'Historical Return', 'Historical Volatility', 'Average Trading Volume', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'Revenue Growth', 'Gross Profit Margin', 'Operating Profit Margin', 'Net Profit Margin', 'Operating Income Growth', 'Net Income Growth', 'Operating Cash Flow', 'Capital Expenditure', 'Free Cash Flow', 'Operating Cash Flow Growth', 'Free Cash Flow Growth', 'FCF Margin']


In [16]:
print("Duplicate company-year rows:", df_merged.duplicated(["SimFinId", "Fiscal Year"]).sum())

Duplicate company-year rows: 0


## Add Balance Sheet

In [17]:
balance_merge = df_balancesheet[['SimFinId', 'Fiscal Year', 'Cash, Cash Equivalents & Short Term Investments', 'Total Current Assets', 'Total Assets', 'Total Current Liabilities', 'Total Liabilities', 'Total Equity', 'balance_diff', 'balance_error_pct', 'Short Term Debt', 'Long Term Debt', 'Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']]

df_merged = df_merged.merge(balance_merge, on=["SimFinId", "Fiscal Year"], how="left", validate="one_to_one")

In [22]:
print(df_merged.shape)

print("Duplicate company-year rows:", df_merged.duplicated(["SimFinId", "Fiscal Year"]).sum())

(386, 45)
Duplicate company-year rows: 0


### Inspect missing values

In [23]:
missing_values = df_merged.isnull().sum()

print(missing_values[missing_values > 0].sort_values(ascending=False))

Free Cash Flow Growth         109
Short Term Debt               107
Equity Growth                  97
Operating Cash Flow Growth     97
Revenue Growth                 97
Operating Income Growth        97
Asset Growth                   97
Debt Growth                    97
Net Income Growth              97
Historical Return              96
Historical Volatility          96
Average Trading Volume         96
Long Term Debt                 31
Gross Profit Margin            27
Gross Profit                   27
Capital Expenditure            14
Free Cash Flow                 14
FCF Margin                     14
dtype: int64


In [24]:
core_missing = df_merged[df_merged[['Long Term Debt', 'Gross Profit', 'Capital Expenditure', 'Free Cash Flow']].isnull().any(axis=1)]

display(core_missing[['Ticker', 'SimFinId', 'Fiscal Year', 'Long Term Debt', 'Gross Profit', 'Capital Expenditure', 'Free Cash Flow']])

,Ticker,SimFinId,Fiscal Year,Long Term Debt,Gross Profit,Capital Expenditure,Free Cash Flow
7,META,121021,2020,NaN,6.927300e+10,-1.511500e+10,2.363200e+10
16,F,249937,2020,1.103410e+11,1.439200e+10,NaN,NaN
20,SPOT,3307751,2020,NaN,2.297668e+09,-8.894199e+07,2.063910e+08
25,CMG,80874,2020,NaN,2.071093e+09,-3.733520e+08,2.904950e+08
26,ISRG,103498,2020,NaN,2.861200e+09,-3.415000e+08,1.143300e+09
...,...,...,...,...,...,...,...
360,MRNA,917766,2023,NaN,2.155000e+09,-7.070000e+08,-3.825000e+09
370,DHIL,7962818,2023,NaN,6.598573e+07,-2.170500e+04,3.464742e+07
371,VCTR,10384115,2023,9.892690e+08,NaN,-5.169000e+06,3.615580e+08
376,AC,919941,2023,NaN,-4.563000e+06,NaN,NaN


In [25]:
feature_columns = [
    'Historical Return',
    'Historical Volatility',
    'Average Trading Volume',
    'Revenue',
    'Gross Profit',
    'Operating Income',
    'Net Income',
    'Revenue Growth',
    'Gross Profit Margin',
    'Operating Profit Margin',
    'Net Profit Margin',
    'Operating Income Growth',
    'Net Income Growth',
    'Operating Cash Flow',
    'Capital Expenditure',
    'Free Cash Flow',
    'Operating Cash Flow Growth',
    'Free Cash Flow Growth',
    'FCF Margin',
    'Cash, Cash Equivalents & Short Term Investments',
    'Total Current Assets',
    'Total Assets',
    'Total Current Liabilities',
    'Total Liabilities',
    'Total Equity',
    'Short Term Debt',
    'Long Term Debt',
    'Current Ratio',
    'Debt-to-Equity',
    'Asset Growth',
    'Equity Growth',
    'Debt Growth'
]

print("Total rows:", len(df_merged))
print("Rows with all features available:", df_merged[feature_columns].notna().all(axis=1).sum())

print("Rows with at least one missing feature:", df_merged[feature_columns].isna().any(axis=1).sum())

Total rows: 386
Rows with all features available: 197
Rows with at least one missing feature: 189


In [26]:
missing_summary = (
    df_merged[feature_columns]
    .isna()
    .sum()
    .to_frame(name='Missing Count')
)

missing_summary['Missing %'] = (missing_summary['Missing Count'] / len(df_merged) * 100)

display(missing_summary.sort_values('Missing %', ascending=False))

,Missing Count,Missing %
Free Cash Flow Growth,109,28.238342
Short Term Debt,107,27.720207
Revenue Growth,97,25.129534
Operating Income Growth,97,25.129534
Operating Cash Flow Growth,97,25.129534
Debt Growth,97,25.129534
Net Income Growth,97,25.129534
Asset Growth,97,25.129534
Equity Growth,97,25.129534
Historical Volatility,96,24.870466


In [28]:
year_missing = (df_merged.groupby('Fiscal Year')[feature_columns].apply(lambda x: x.isna().sum()))

display(year_missing)

,Historical Return,Historical Volatility,Average Trading Volume,Revenue,Gross Profit,Operating Income,Net Income,Revenue Growth,Gross Profit Margin,Operating Profit Margin,...,Total Current Liabilities,Total Liabilities,Total Equity,Short Term Debt,Long Term Debt,Current Ratio,Debt-to-Equity,Asset Growth,Equity Growth,Debt Growth
Fiscal Year,,,,,,,,,,,,,,,,,,,,,
2020,89,89,89,0,7,0,0,89,7,0,...,0,0,0,28,9,0,0,89,89,89
2021,7,7,7,0,7,0,0,8,7,0,...,0,0,0,29,8,0,0,8,8,8
2022,0,0,0,0,6,0,0,0,6,0,...,0,0,0,26,8,0,0,0,0,0
2023,0,0,0,0,6,0,0,0,6,0,...,0,0,0,22,6,0,0,0,0,0
2024,0,0,0,0,1,0,0,0,1,0,...,0,0,0,2,0,0,0,0,0,0


In [29]:
print("Final shape:", df_merged.shape)

print("Duplicate company-year rows:", df_merged.duplicated(['SimFinId', 'Fiscal Year']).sum())

print("Unique companies:", df_merged['SimFinId'].nunique())

print("Fiscal years:", sorted(df_merged['Fiscal Year'].unique()))

Final shape: (386, 45)
Duplicate company-year rows: 0
Unique companies: 97
Fiscal years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


### Save to /content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/merged/

In [30]:
df_merged.to_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/merged/merged_dataset.csv", index=False)

print("Saved:", "/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/merged/merged_dataset.csv")

Saved: /content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/merged/merged_dataset.csv
